# 획 추출 기반 후보 → 실제 추가 역확산 (L4)

1 → 2 → 3 순서로 실행합니다. 기존 `sweep_...zip` 하나만 있으면 됩니다.
Drive 마운트와 best.pt 재탐색, 획 추출 재실행이 필요 없습니다.
2번의 `ZIP_PATH`에 이미 올린 파일 경로를 쓰거나, 비워두고 ZIP을 업로드하세요.

기본 사례는 strength=.65 / seed=100입니다. 원래 생성물을 다시 노이즈화하고,
같은 초기 latent로 **추가 역확산만** 한 결과와 **획 추출 기반 후보로 유도한 역확산**을 비교합니다.
기하학적 후보는 가이드에 사용하지 않습니다. 후보는 정답 이미지가 아니며,
후보와의 일치도 향상 자체가 실제 오류 감소나 HSQM 성능을 보장하지는 않습니다.
최종 PNG는 VAE를 거친 실제 역확산 출력이며 후보 픽셀을 붙여 넣지 않습니다.

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install',
    'diffusers==0.37.0', 'transformers==4.57.6', 'accelerate',
    'safetensors', 'numpy', 'pillow', 'matplotlib'], check=True)
from pathlib import Path
import importlib.util
HELPER_SOURCE = '"""Paired DDIM re-denoising guided by an extractor-fitted candidate, not GT."""\nimport hashlib\nimport io\nimport json\nfrom pathlib import Path\nimport zipfile\n\nimport numpy as np\nfrom PIL import Image\nimport torch\nfrom torch.nn import functional as F\n\n\ndef load_case(archive, case_id):\n    """Read named members only; recorded Colab absolute paths are not trusted."""\n    with zipfile.ZipFile(archive) as z:\n        cases = json.loads(z.read(\'case_manifest.json\'))\n        matches = [c for c in cases if c[\'case_id\'] == case_id]\n        if len(matches) != 1:\n            raise ValueError(\'Select exactly one case from case_manifest.json\')\n        case = matches[0]\n        page = f"generation/page_{case[\'page_id\']:03d}.png"\n        candidate = f"{case_id}/normal_candidate.png"\n        def read_image(name):\n            with Image.open(io.BytesIO(z.read(name))) as im:\n                return im.convert(\'RGB\').copy()\n        source, target = read_image(page), read_image(candidate)\n        if source.size != target.size or any(n % 8 for n in source.size):\n            raise ValueError(\'Input and fitted candidate must have equal dimensions divisible by 8\')\n        report = json.loads(z.read(\'generation/generation_report.json\'))\n        metadata = dict(case=case, candidate_member=candidate, input_member=page,\n                        generation_model=report[\'model_id\'],\n                        candidate_is_ground_truth=False,\n                        prompt=report.get(\'prompts\', {}).get(\'4\', \'\'),\n                        negative_prompt=report.get(\'negative_prompt\', \'\'))\n    return source, target, metadata\n\n\ndef image_tensor(image, device):\n    a = np.asarray(image, dtype=np.float32) / 255\n    return torch.from_numpy(a.copy()).permute(2, 0, 1)[None].to(device)\n\n\nclass CandidateObjective:\n    """Fixed spatial weights; gradients stay continuous through the VAE."""\n    def __init__(self, source, candidate, threshold=.10, radius=2, preserve_weight=.25):\n        if source.shape != candidate.shape or not 0 < threshold < 1 or radius < 0:\n            raise ValueError(\'Invalid images or edit-region settings\')\n        self.source, self.candidate = source.detach(), candidate.detach()\n        difference = (source - candidate).abs().mean(1, keepdim=True)\n        edit = (difference > threshold).float()\n        self.edit = F.max_pool2d(edit, 2*radius+1, 1, radius).detach()\n        self.preserve_weight = preserve_weight\n\n    @staticmethod\n    def mean_region(values, mask):\n        return (values*mask).sum() / (mask.sum()*values.shape[1]).clamp_min(1)\n\n    def __call__(self, image):\n        match = self.mean_region((image-self.candidate).square(), self.edit)\n        preserve = self.mean_region((image-self.source).square(), 1-self.edit)\n        loss = match + self.preserve_weight*preserve\n        return loss, dict(candidate_region_mse=float(match.detach()),\n                          preserved_region_mse=float(preserve.detach()))\n\n\ndef optimize_clean(clean, decode, objective, *, iterations=4, step_ratio=.02,\n                   budget_ratio=.08, backtracks=5):\n    """Backtrack on the SAME clean estimate; bound total correction per step."""\n    anchor = clean.detach().float()\n    current = anchor.clone()\n    scale = anchor.square().mean().sqrt().clamp_min(1e-3)\n    with torch.no_grad():\n        before, _ = objective(decode(current))\n    accepted = 0\n    for _ in range(iterations):\n        with torch.enable_grad():\n            leaf = current.detach().requires_grad_(True)\n            loss, _ = objective(decode(leaf))\n            gradient, = torch.autograd.grad(loss, leaf)\n        if not torch.isfinite(gradient).all() or not torch.isfinite(loss):\n            raise FloatingPointError(\'Nonfinite candidate guidance\')\n        norm = gradient.square().mean().sqrt()\n        if norm < 1e-12:\n            break\n        direction = step_ratio*scale*gradient/norm\n        found = False\n        with torch.no_grad():\n            for attempt in range(backtracks+1):\n                proposal = current-direction*(.5**attempt)\n                delta = proposal-anchor\n                rms = delta.square().mean().sqrt().clamp_min(1e-12)\n                proposal = anchor+delta*(budget_ratio*scale/rms).clamp(max=1)\n                trial, _ = objective(decode(proposal))\n                if torch.isfinite(trial) and trial < loss.detach():\n                    current = proposal.detach()\n                    accepted += 1\n                    found = True\n                    break\n        if not found:\n            break\n    with torch.no_grad():\n        after, details = objective(decode(current))\n    return current, dict(loss_before=float(before), loss_after=float(after),\n                         accepted_inner_steps=accepted, **details)\n\n\ndef run_pair(source, candidate, metadata, output, *, steps=60, strength=.35,\n             seed=2026, cfg_scale=5., inner_steps=4, step_ratio=.02,\n             model_id=\'stable-diffusion-v1-5/stable-diffusion-v1-5\', revision=None,\n             progress=None):\n    """Identical source VAE encoding, noise, prompt, scheduler in both arms."""\n    if not torch.cuda.is_available():\n        raise RuntimeError(\'Select an L4 GPU runtime in Colab\')\n    if not 0 < strength <= 1 or steps < 4 or int(steps*strength) < 2:\n        raise ValueError(\'Need at least two actual re-denoising steps\')\n    if inner_steps < 1 or step_ratio <= 0 or cfg_scale <= 1:\n        raise ValueError(\'Invalid guidance configuration\')\n    from diffusers import StableDiffusionImg2ImgPipeline, DDIMScheduler\n    from copy import deepcopy\n    out = Path(output)\n    out.mkdir(parents=True, exist_ok=False)\n    source.save(out/\'input.png\')\n    candidate.save(out/\'fitted_candidate.png\')\n    kwargs = dict(torch_dtype=torch.float16, use_safetensors=True)\n    if revision:\n        kwargs[\'revision\'] = revision\n    pipe = StableDiffusionImg2ImgPipeline.from_pretrained(model_id, **kwargs)\n    pipe.scheduler = DDIMScheduler.from_config(pipe.scheduler.config,\n                                              clip_sample=False, thresholding=False)\n    pipe.to(\'cuda\')\n    pipe.enable_attention_slicing()\n    pipe.enable_vae_slicing()\n    for module in (pipe.unet, pipe.vae, pipe.text_encoder):\n        module.eval().requires_grad_(False)\n    prompt = metadata.get(\'prompt\') or (\'four Korean Hangul glyphs, black ink on white, \'\n                                      \'preserve the input typography and layout\')\n    with torch.no_grad():\n        positive, negative = pipe.encode_prompt(prompt, \'cuda\', 1, True,\n            negative_prompt=metadata.get(\'negative_prompt\', \'\'))\n        embeddings = torch.cat([negative, positive])\n        pipe.scheduler.set_timesteps(steps, device=\'cuda\')\n        timesteps, _ = pipe.get_timesteps(steps, strength, \'cuda\')\n        # VAE and noise initialization exactly once, then copy for both arms.\n        initial = pipe.prepare_latents(pipe.image_processor.preprocess(source), timesteps[:1],\n            1, 1, embeddings.dtype, \'cuda\', torch.Generator(\'cpu\').manual_seed(seed))\n    pipe.vae.to(dtype=torch.float32)\n    def decode(z):\n        return pipe.vae.decode(z.float()/pipe.vae.config.scaling_factor,\n                               return_dict=False)[0].float()/2+.5\n    def save_image(z, path):\n        with torch.no_grad():\n            array = decode(z)[0].permute(1, 2, 0).clamp(0, 1).cpu().numpy()\n        Image.fromarray(np.rint(array*255).astype(\'uint8\')).save(path)\n    objective = CandidateObjective(image_tensor(source, \'cuda\'), image_tensor(candidate, \'cuda\'))\n    Image.fromarray((objective.edit[0, 0].cpu().numpy()*255).astype(\'uint8\')).save(out/\'edit_region.png\')\n    traces, metrics = {}, {}\n    for method in (\'redenoise_only\', \'candidate_guided\'):\n        scheduler = deepcopy(pipe.scheduler)\n        latent = initial.detach().float().clone()\n        trace = []\n        for i, t in enumerate(timesteps):\n            with torch.no_grad():\n                inputs = scheduler.scale_model_input(torch.cat([latent, latent]), t)\n                u, c = pipe.unet(inputs.to(torch.float16), t,\n                    encoder_hidden_states=embeddings, return_dict=False)[0].chunk(2)\n                noise = (u+cfg_scale*(c-u)).float()\n                base = scheduler.step(noise, t, latent, eta=0., return_dict=True)\n            record = dict(step=i, timestep=int(t), active=False, accepted_inner_steps=0)\n            latent = base.prev_sample.detach()\n            clean = base.pred_original_sample.detach()\n            if method == \'candidate_guided\' and i/max(len(timesteps)-1, 1) >= .2:\n                clean, result = optimize_clean(clean, decode, objective,\n                    iterations=inner_steps if i < len(timesteps)-1 else inner_steps*2,\n                    step_ratio=step_ratio)\n                # DDIM eta=0: x_(t-1) = sqrt(a_prev)*x0 + sqrt(1-a_prev)*epsilon.\n                # Hold the predicted epsilon fixed and change the clean component.\n                prev_t = int(t)-scheduler.config.num_train_timesteps//scheduler.num_inference_steps\n                alpha = (scheduler.alphas_cumprod[prev_t] if prev_t >= 0\n                         else scheduler.final_alpha_cumprod).to(latent)\n                latent = latent+alpha.sqrt()*(clean-base.pred_original_sample)\n                record.update(active=True, **result)\n            latent = latent.detach()\n            trace.append(record)\n            if i % 4 == 0 or i == len(timesteps)-1:\n                save_image(clean, out/f\'{method}_step_{i:03d}.png\')\n            if progress:\n                progress(method, i+1, len(timesteps), record)\n        save_image(latent, out/f\'{method}.png\')\n        with torch.no_grad():\n            loss, values = objective(decode(latent))\n        metrics[method] = dict(loss=float(loss), **values)\n        traces[method] = trace\n    with torch.no_grad():\n        loss, values = objective(objective.source)\n    metrics[\'input\'] = dict(loss=float(loss), **values)\n    report = dict(metadata=metadata, model_id=model_id, revision=revision,\n        steps=steps, actual_steps=len(timesteps), strength=strength, seed=seed,\n        cfg_scale=cfg_scale, inner_steps=inner_steps, step_ratio=step_ratio,\n        initial_latent_sha256=hashlib.sha256(initial.cpu().numpy().tobytes()).hexdigest(),\n        paired_initial_latent=True, guidance_target=\'extractor_fitted_candidate\',\n        target_pixels_pasted_into_output=False, metrics=metrics, traces=traces,\n        metric_note=\'Candidate agreement, not ground-truth accuracy or HSQM score\')\n    (out/\'report.json\').write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding=\'utf-8\')\n    del pipe\n    torch.cuda.empty_cache()\n    return report\n'
helper_path=Path('/content/fitted_candidate_diffusion_runtime.py')
helper_path.write_text(HELPER_SOURCE,encoding='utf-8')
spec=importlib.util.spec_from_file_location('fitted_candidate_diffusion_runtime',helper_path)
repair=importlib.util.module_from_spec(spec);spec.loader.exec_module(repair)
import torch
if not torch.cuda.is_available():raise RuntimeError('런타임 → 런타임 유형 변경 → L4 GPU를 선택하세요.')
print('GPU:',torch.cuda.get_device_name(0))

In [ ]:
import json, zipfile
from google.colab import files
from IPython.display import display

ZIP_PATH=''  # 예: '/content/sweep_20261111_062740_406072.zip'
CASE_ID='case_003_strength_0p65_seed_100'
if not ZIP_PATH:
    uploaded=files.upload()
    names=[name for name in uploaded if name.lower().endswith('.zip')]
    if len(names)!=1:raise ValueError('sweep ZIP 하나를 선택하세요.')
    ZIP_PATH=names[0]
with zipfile.ZipFile(ZIP_PATH) as z:
    available=json.loads(z.read('case_manifest.json'))
print('선택 가능한 사례:',[c['case_id'] for c in available])
source,candidate,metadata=repair.load_case(ZIP_PATH,CASE_ID)
print('선택:',CASE_ID,'| 왼쪽: 오류 입력 / 오른쪽: 획 추출 기반 후보')
import matplotlib.pyplot as plt
fig,axes=plt.subplots(1,2,figsize=(10,5))
for ax,im,title in zip(axes,[source,candidate],['Erroneous input','Extractor-fitted guide']):
    ax.imshow(im);ax.set_title(title);ax.axis('off')
plt.show();plt.close(fig)

In [ ]:
from datetime import datetime
import shutil
from PIL import Image
from huggingface_hub import model_info

# .35는 이번 추가 역확산의 강도입니다. 원래 오류 생성 강도 .65와 별개입니다.
STRENGTH=.35
STEPS=60  # 실제 추가 역확산은 int(STEPS*STRENGTH)=21단계
INNER_STEPS=4
STEP_RATIO=.02
SEED=2026
MODEL_ID='stable-diffusion-v1-5/stable-diffusion-v1-5'
REVISION=model_info(MODEL_ID).sha
OUTPUT=Path('/content')/('fitted_reverse_'+datetime.now().strftime('%Y%m%d_%H%M%S_%f'))
def progress(method,step,total,record):
    if step==1 or step%4==0 or step==total:
        print(f"{method}: {step}/{total} | accepted={record['accepted_inner_steps']} | "
              f"loss={record.get('loss_before')} → {record.get('loss_after')}",flush=True)
report=repair.run_pair(source,candidate,metadata,OUTPUT,steps=STEPS,strength=STRENGTH,
    seed=SEED,inner_steps=INNER_STEPS,step_ratio=STEP_RATIO,
    model_id=MODEL_ID,revision=REVISION,progress=progress)
panels=[('Input',source),('Re-denoise only',Image.open(OUTPUT/'redenoise_only.png')),
        ('Candidate-guided reverse diffusion',Image.open(OUTPUT/'candidate_guided.png')),
        ('Guide ONLY',candidate)]
fig,axes=plt.subplots(1,4,figsize=(20,5))
for ax,(title,im) in zip(axes,panels):
    ax.imshow(im);ax.set_title(title);ax.axis('off')
fig.tight_layout();fig.savefig(OUTPUT/'comparison.png',dpi=140);plt.show();plt.close(fig)
print('실제 추가 단계:',report['actual_steps'])
print('후보에 대한 손실(정확도 아님):',json.dumps(report['metrics'],indent=2))
for method in ('redenoise_only','candidate_guided'):
    snapshots=sorted(OUTPUT.glob(method+'_step_*.png'))
    fig,axes=plt.subplots(1,len(snapshots),figsize=(4*len(snapshots),4),squeeze=False)
    for ax,path in zip(axes.flat,snapshots):
        ax.imshow(Image.open(path));ax.set_title(path.stem);ax.axis('off')
    fig.tight_layout();fig.savefig(OUTPUT/(method+'_trajectory.png'),dpi=120);plt.show();plt.close(fig)
(OUTPUT/'runtime_source.py').write_text(HELPER_SOURCE,encoding='utf-8')
archive=shutil.make_archive(str(OUTPUT),'zip',OUTPUT)
print('결과:',archive)
files.download(archive)